# 🚗 RTA Dataset — Preprocessing & Modelisation ML
## Notebook 02 — Final

---

| Champ | Valeur |
|-------|--------|
| **Auteur** | [Votre Nom] |
| **Module** | Machine Learning — M. Abdallah Khemais |
| **Dataset** | RTA Dataset (Road Traffic Accidents) |
| **Tache** | Classification multiclasse — `Accident_severity` |

---

## Plan
1. Imports
2. Chargement
3. Preprocessing & Pipeline
4. Feature Engineering
5. Split Train/Test
6. Modelisation — 3 algorithmes
7. Evaluation multi-metriques
8. GridSearchCV
9. Analyse des erreurs
10. Importance des features
11. Conclusion

---

## Pourquoi l'Accuracy seule est insuffisante ici ?

Le dataset est **fortement desequilibre** : ~72% Slight Injury, ~24% Serious, ~4% Fatal.  
Un modele qui predit toujours 'Slight Injury' obtiendrait **72% d'accuracy sans rien apprendre**.  
On utilise donc plusieurs metriques complementaires :

| Metrique | Ce qu'elle mesure | Pertinence ici |
|----------|-------------------|----------------|
| **Accuracy** | % de bonnes predictions global | Fournie mais insuffisante seule |
| **F1-macro** | Moyenne F1 par classe (sans ponderation) | **Principale** — penalise les classes ignorees |
| **Precision macro** | Quand on predit X, est-ce correct ? | Eviter les fausses alarmes |
| **Recall macro** | Parmi les vrais X, combien detectes ? | Crucial pour Fatal Injury |
| **ROC-AUC (OvR)** | Capacite de discrimination entre classes | Vue globale independante du seuil |

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import (
    LabelEncoder, StandardScaler,
    OrdinalEncoder, label_binarize
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeles
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Metriques — TOUTES les metriques necessaires
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
sns.set_palette('husl')

print('Imports OK')

Imports OK


## 2. Chargement des Donnees

On recharge depuis le dataset original en appliquant le **shuffle sur les 12 300 lignes** puis on prend 3 000 lignes.  
Cela garantit que l'echantillon est representatif de l'ensemble du dataset.

In [2]:
try:
    # Chargement depuis la sortie de l'EDA
    df = pd.read_csv('../Data/df_eda_ready.csv')
    print('Chargement depuis df_eda_ready.csv (EDA deja fait)')
except FileNotFoundError:
    # Fallback : rechargement depuis l'original avec shuffle complet
    df_full = pd.read_csv('../data/RTA_Dataset.csv')
    print(f'Dataset complet : {df_full.shape[0]} lignes')
    # Shuffle sur TOUTES les lignes avant echantillonnage
    df_full = df_full.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    df = df_full.head(3000).copy()
    print(f'Shuffle sur {df_full.shape[0]} lignes => echantillon de 3 000')

TARGET = 'Accident_severity'
print(f'Dimensions : {df.shape}')
print(f'Classes    : {df[TARGET].unique()}')
df.head()

Chargement depuis df_eda_ready.csv (EDA deja fait)
Dimensions : (3000, 32)
Classes    : ['Slight Injury' 'Serious Injury' 'Fatal injury']


,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,14:27:00,Friday,Over 51,Male,Junior high school,Employee,Above 10yr,NaN,Owner,2-5yrs,...,U-Turn,na,na,na,na,Driver,Normal,Not a Pedestrian,Changing lane to the left,Slight Injury
1,16:00:00,Sunday,Under 18,Male,Junior high school,Employee,Below 1yr,Automobile,Owner,NaN,...,Other,na,na,na,na,NaN,NaN,Not a Pedestrian,Driving carelessly,Slight Injury
2,16:56:00,Wednesday,18-30,Male,NaN,NaN,NaN,NaN,NaN,NaN,...,Getting off,Driver or rider,Male,18-30,3,Driver,Normal,Not a Pedestrian,No distancing,Serious Injury
3,15:48:00,Tuesday,31-50,Male,Junior high school,Employee,Below 1yr,Public (12 seats),Owner,NaN,...,Going straight,Driver or rider,Male,18-30,3,Self-employed,Normal,Not a Pedestrian,Moving Backward,Slight Injury
4,12:48:00,Friday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,Unknown,...,Going straight,Passenger,Female,31-50,3,Driver,Normal,Not a Pedestrian,No distancing,Slight Injury


## 3. Preprocessing & Pipeline sklearn

### Regles appliquees
- **Aucune ligne supprimee** : les valeurs manquantes sont imputees dans le pipeline
- **Mediane** pour les numeriques (robuste aux outliers)
- **Mode** pour les categorielles
- **StandardScaler** pour normaliser les numeriques
- **OrdinalEncoder** pour encoder les categorielles

In [3]:
# Encodage de la variable cible
le_target = LabelEncoder()
y = le_target.fit_transform(df[TARGET])
X = df.drop(columns=[TARGET])

print('Mapping cible :')
for orig, enc in zip(le_target.classes_, le_target.transform(le_target.classes_)):
    print(f'  {orig:<30} => {enc}')

# Types de colonnes
numerical_features   = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(f'\nFeatures numeriques    : {len(numerical_features)}')
print(f'Features categorielles : {len(categorical_features)}')

Mapping cible :
  Fatal injury                   => 0
  Serious Injury                 => 1
  Slight Injury                  => 2

Features numeriques    : 2
Features categorielles : 29


In [4]:
# Pipeline numerique
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # mediane : robuste aux outliers
    ('scaler',  StandardScaler())                    # normalisation
])

# Pipeline categoriel
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # mode
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Assemblage
preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Pipeline defini :')
print('  num -> SimpleImputer(median) + StandardScaler')
print('  cat -> SimpleImputer(most_frequent) + OrdinalEncoder')
print('  Aucune ligne supprimee — toutes les donnees conservees')

Pipeline defini :
  num -> SimpleImputer(median) + StandardScaler
  cat -> SimpleImputer(most_frequent) + OrdinalEncoder
  Aucune ligne supprimee — toutes les donnees conservees


## 4. Feature Engineering

In [5]:
X = X.copy()

# Feature 1 : Periode de la journee
# L'heure brute (0-23) est moins lisible pour le modele que la periode
if 'Time' in X.columns:
    X['Hour'] = pd.to_datetime(
        X['Time'], format='%H:%M:%S', errors='coerce'
    ).dt.hour
    X['Time_Period'] = pd.cut(
        X['Hour'].fillna(0),
        bins=[0, 6, 12, 18, 24],
        labels=['Nuit','Matin','Apres-midi','Soir'],
        right=False
    ).astype(str)
    categorical_features.append('Time_Period')
    print('Feature creee : Time_Period')

# Feature 2 : Interaction Jour x Luminosite
if 'Day_of_week' in X.columns and 'Light_conditions' in X.columns:
    X['Day_Light'] = (
        X['Day_of_week'].astype(str) + '_' +
        X['Light_conditions'].astype(str)
    )
    categorical_features.append('Day_Light')
    print('Feature creee : Day_Light')

print(f'Total features : {len(X.columns)}')

Feature creee : Time_Period
Feature creee : Day_Light
Total features : 34


## 5. Split Train / Test

In [6]:
# stratify=y : chaque classe garde sa proportion dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train : {X_train.shape[0]} exemples ({80}%)')
print(f'Test  : {X_test.shape[0]}  exemples ({20}%)')
print('\nVerification stratification :')
print('  Train :', {le_target.classes_[k]: v for k, v in zip(*np.unique(y_train, return_counts=True))})
print('  Test  :', {le_target.classes_[k]: v for k, v in zip(*np.unique(y_test,  return_counts=True))})

Train : 2400 exemples (80%)
Test  : 600  exemples (20%)

Verification stratification :
  Train : {'Fatal injury': np.int64(36), 'Serious Injury': np.int64(356), 'Slight Injury': np.int64(2008)}
  Test  : {'Fatal injury': np.int64(9), 'Serious Injury': np.int64(89), 'Slight Injury': np.int64(502)}


## 6. Modelisation — 3 Algorithmes

| Algorithme | Type | Justification |
|------------|------|---------------|
| Regression Logistique | Lineaire | Baseline, rapide, interpretable |
| Arbre de Decision | Non-lineaire | Interpretable, gere bien les categorielles |
| Random Forest | Ensembliste | Robuste, reduit la variance par bagging |

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000, class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            max_depth=8, class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}
print('3 pipelines prets.')

3 pipelines prets.


## 7. Evaluation Multi-Metriques

On calcule **5 metriques** pour chaque modele :  
`Accuracy`, `F1-macro`, `Precision-macro`, `Recall-macro`, `ROC-AUC (OvR)`

**Rappel des definitions :**
- **Accuracy** : (TP + TN) / Total — trompeuse si classes desequilibrees
- **Precision** : TP / (TP + FP) — quand je dis 'Fatal', ai-je raison ?
- **Recall** : TP / (TP + FN) — parmi tous les accidents fatals, combien ai-je detectes ?
- **F1-macro** : harmonie Precision/Recall, moyenne sur toutes les classes sans poids
- **ROC-AUC** : capacite de discrimination — 1.0 = parfait, 0.5 = aleatoire

In [8]:
def evaluate_model(pipe, X_tr, y_tr, X_te, y_te, cv, name):
    """Entraine le modele et calcule toutes les metriques."""
    # Validation croisee sur le train
    cv_f1 = cross_val_score(
        pipe, X_tr, y_tr, cv=cv,
        scoring='f1_macro', n_jobs=-1
    )

    # Entrainement final
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)

    # Probabilites pour AUC
    try:
        y_proba = pipe.predict_proba(X_te)
        auc = roc_auc_score(y_te, y_proba, multi_class='ovr', average='macro')
    except Exception:
        auc = np.nan

    return {
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std':  cv_f1.std(),
        'accuracy':   accuracy_score(y_te, y_pred),
        'f1_macro':   f1_score(y_te, y_pred, average='macro'),
        'precision':  precision_score(y_te, y_pred, average='macro', zero_division=0),
        'recall':     recall_score(y_te, y_pred, average='macro', zero_division=0),
        'auc':        auc,
        'pred':       y_pred,
        'pipe':       pipe
    }

# --- Entrainement de tous les modeles ---
results = {}

print(f'{"Modele":<25} {"CV F1":>8} {"Acc":>7} {"F1":>7} {"Prec":>7} {"Rec":>7} {"AUC":>7}')
print('-' * 76)

for name, pipe in models.items():
    res = evaluate_model(pipe, X_train, y_train, X_test, y_test, cv, name)
    results[name] = res
    print(
        f'{name:<25}'
        f' {res["cv_f1_mean"]:.4f}'
        f' {res["accuracy"]:.4f}'
        f' {res["f1_macro"]:.4f}'
        f' {res["precision"]:.4f}'
        f' {res["recall"]:.4f}'
        f' {res["auc"]:.4f}'
    )

Modele                       CV F1     Acc      F1    Prec     Rec     AUC
----------------------------------------------------------------------------
Logistic Regression       0.3188 0.5217 0.3377 0.3698 0.5216 0.6083
Decision Tree             0.3621 0.7300 0.4399 0.4217 0.4862 0.6066
Random Forest             0.3037 0.8367 0.3037 0.2789 0.3333 0.6895


In [ ]:
# --- Tableau comparatif complet ---
comp = pd.DataFrame([
    {
        'Modele':     k,
        'CV F1-macro': v['cv_f1_mean'],
        'Accuracy':    v['accuracy'],
        'F1-macro':    v['f1_macro'],
        'Precision':   v['precision'],
        'Recall':      v['recall'],
        'ROC-AUC':     v['auc']
    }
    for k, v in results.items()
]).set_index('Modele')

print('=== TABLEAU COMPARATIF MULTI-METRIQUES ===')
print(comp.round(4).to_string())

# Visualisation
metriques = ['Accuracy','F1-macro','Precision','Recall','ROC-AUC']
comp[metriques].plot(
    kind='bar', figsize=(13, 5),
    edgecolor='black', colormap='tab10'
)
plt.title('Comparaison Multi-Metriques des 3 Modeles',
          fontweight='bold', fontsize=13)
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.axhline(y=0.72, color='red', linestyle='--', alpha=0.5,
            label='Accuracy naive (toujours Slight = 72%)')
plt.tight_layout()
plt.show()

best_base = max(results, key=lambda k: results[k]['f1_macro'])
print(f'\nMeilleur modele avant optimisation (F1-macro) : {best_base}')

In [ ]:
# --- Rapports de classification detailles ---
for name, res in results.items():
    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    print(classification_report(
        y_test, res['pred'],
        target_names=le_target.classes_
    ))

In [ ]:
# --- Matrices de confusion ---
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['pred'])
    ConfusionMatrixDisplay(cm, display_labels=le_target.classes_).plot(
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(
        f'{name}\nF1={res["f1_macro"]:.3f}  AUC={res["auc"]:.3f}',
        fontweight='bold', fontsize=10
    )
    ax.tick_params(axis='x', rotation=25, labelsize=8)

plt.suptitle('Matrices de Confusion — 3 Modeles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Optimisation — GridSearchCV

### Pourquoi GridSearchCV ?

- **GridSearchCV** : teste **toutes les combinaisons** de la grille de facon exhaustive → resultat garanti optimal sur l'espace explore
- **RandomizedSearchCV** : tire aleatoirement → peut rater la meilleure combinaison

On garde une grille **ciblée et raisonnee** pour que l'exploration exhaustive reste faisable.  
La metrique d'optimisation est **F1-macro** (pas l'accuracy) car le dataset est desequilibre.

In [ ]:
# Grille d'hyperparametres — chaque combinaison sera testee
param_grid = {
    # Nombre d'arbres : plus = plus robuste, mais plus lent
    'classifier__n_estimators':      [50, 100, 150],
    # Profondeur : None = arbre complet (risque overfitting), 8/12 = regularisation
    'classifier__max_depth':         [None, 8, 12],
    # Echantillons min pour diviser : plus grand = plus de regularisation
    'classifier__min_samples_split': [2, 5, 10],
    # Features par split : sqrt et log2 = sous-ensemble aleatoire (diversite des arbres)
    'classifier__max_features':      ['sqrt', 'log2']
}

total_combinations = 1
for v in param_grid.values():
    total_combinations *= len(v)

print(f'Combinaisons : {total_combinations}')
print(f'Total fits   : {total_combinations} x 5 folds = {total_combinations * 5}\n')

grid_search = GridSearchCV(
    estimator  = results['Random Forest']['pipe'],
    param_grid = param_grid,
    cv         = cv,
    scoring    = 'f1_macro',   # on optimise F1-macro, PAS l'accuracy
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

print('Lancement GridSearchCV...')
grid_search.fit(X_train, y_train)

print('\nMeilleurs hyperparametres :')
for p, v in grid_search.best_params_.items():
    print(f'  {p:<40} : {v}')
print(f'\n  => Best CV F1-macro : {grid_search.best_score_:.4f}')

In [ ]:
# --- Evaluation du modele optimise ---
best_model = grid_search.best_estimator_
y_best     = best_model.predict(X_test)

try:
    y_best_proba = best_model.predict_proba(X_test)
    auc_opt = roc_auc_score(y_test, y_best_proba, multi_class='ovr', average='macro')
except Exception:
    auc_opt = np.nan

acc_opt  = accuracy_score(y_test, y_best)
f1_opt   = f1_score(y_test, y_best, average='macro')
prec_opt = precision_score(y_test, y_best, average='macro', zero_division=0)
rec_opt  = recall_score(y_test, y_best, average='macro', zero_division=0)

# Avant / apres
rf_before = results['Random Forest']
print('=== AVANT / APRES GridSearchCV — Random Forest ===')
print(f'{"Metrique":<15} {"Avant":>8} {"Apres":>8} {"Gain":>8}')
print('-' * 44)
for label, before, after in [
    ('Accuracy',  rf_before['accuracy'],  acc_opt),
    ('F1-macro',  rf_before['f1_macro'],  f1_opt),
    ('Precision', rf_before['precision'], prec_opt),
    ('Recall',    rf_before['recall'],    rec_opt),
    ('ROC-AUC',   rf_before['auc'],       auc_opt),
]:
    print(f'{label:<15} {before:>8.4f} {after:>8.4f} {after-before:>+8.4f}')

print(f'\nRapport de classification final :')
print(classification_report(y_test, y_best, target_names=le_target.classes_))

In [ ]:
# --- Visualisation avant / apres ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, rf_before['pred']),
    display_labels=le_target.classes_
).plot(ax=axes[0], cmap='Oranges', colorbar=False)
axes[0].set_title(
    f'RF avant GridSearch\nF1={rf_before["f1_macro"]:.3f}  AUC={rf_before["auc"]:.3f}',
    fontweight='bold'
)
axes[0].tick_params(axis='x', rotation=25)

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_best),
    display_labels=le_target.classes_
).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(
    f'RF apres GridSearch\nF1={f1_opt:.3f}  AUC={auc_opt:.3f}',
    fontweight='bold'
)
axes[1].tick_params(axis='x', rotation=25)

plt.suptitle('Impact de GridSearchCV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 10 combinaisons de la grille ---
cv_res = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
cols = ['rank_test_score','mean_test_score','std_test_score',
        'param_classifier__n_estimators','param_classifier__max_depth',
        'param_classifier__min_samples_split','param_classifier__max_features']
print('Top 10 combinaisons testees :')
print(cv_res[cols].head(10).to_string(index=False))

## 9. Analyse des Erreurs

In [ ]:
# Distribution des erreurs par classe reelle
error_mask = y_test != y_best
n_errors   = error_mask.sum()
print(f'Erreurs totales : {n_errors} / {len(y_test)} ({n_errors/len(y_test)*100:.1f}%)')

errors_real = le_target.inverse_transform(y_test[error_mask])
errors_pred = le_target.inverse_transform(y_best[error_mask])

error_df = pd.DataFrame({'Reel': errors_real, 'Predit': errors_pred})
print('\nErreurs par classe reelle :')
print(error_df['Reel'].value_counts().to_string())

print('\nConcentration des confusions (reel => predit) :')
print(error_df.groupby(['Reel','Predit']).size()
             .sort_values(ascending=False).head(10).to_string())

## 10. Importance des Features

In [ ]:
rf_clf      = best_model.named_steps['classifier']
importances = rf_clf.feature_importances_
n           = len(importances)
feat_names  = (numerical_features + categorical_features)[:n]

imp_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})\
           .sort_values('Importance', ascending=False)

print('Top 15 features :')
print(imp_df.head(15).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(15), x='Importance', y='Feature',
            palette='viridis', edgecolor='black')
plt.title('Top 15 Features — RF Optimise par GridSearchCV',
          fontweight='bold', fontsize=12)
plt.xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

## 11. Conclusion

### Bilan des Performances

| Modele | Accuracy | F1-macro | Precision | Recall | ROC-AUC |
|--------|:--------:|:--------:|:---------:|:------:|:-------:|
| Logistic Regression | a completer | a completer | a completer | a completer | a completer |
| Decision Tree | a completer | a completer | a completer | a completer | a completer |
| Random Forest | a completer | a completer | a completer | a completer | a completer |
| **RF + GridSearchCV** | **a completer** | **a completer** | **a completer** | **a completer** | **a completer** |

> Remplir ce tableau apres execution complete.

### Enseignements cles

1. **Accuracy seule = insuffisante** sur donnees desequilibrees. Un modele naif aurait ~72% d'accuracy en ne predisant que 'Slight Injury'. On utilise F1-macro, Precision, Recall et ROC-AUC.
2. **Shuffle sur 12 300 lignes** avant echantillonnage = echantillon representatif sans biais d'ordre.
3. **Valeurs manquantes imputees** dans le pipeline, pas supprimees — conservation maximale de l'information.
4. **GridSearchCV** : exploration exhaustive garantit la meilleure combinaison sur la grille testee.
5. **class_weight='balanced'** : indispensable pour que le modele accorde de l'importance a Fatal Injury (~4%).
6. **Random Forest** : meilleur modele grace au bagging (agregation de plusieurs arbres independants).

### Limites
- Fatal Injury tres sous-represente : recall faible sur cette classe meme avec `class_weight='balanced'`
- Dataset localise a Addis Abeba : generalisation geographique non garantie
- Pas de donnees temps-reel : vitesse, GPS, meteo

### Perspectives
- SMOTE pour synthetiser des exemples de la classe minoritaire
- XGBoost / LightGBM
- Deploiement Streamlit

In [ ]:
os.makedirs('../src', exist_ok=True)
with open('../src/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../src/label_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f)

print('=' * 62)
print('RESUME FINAL')
print('=' * 62)
print(f'Dataset         : RTA ({df.shape[0]} lignes, shuffle sur 12300)')
print(f'Valeurs NaN     : Imputees (mediane/mode) — rien supprime')
print(f'Optimisation    : GridSearchCV ({total_combinations} comb. x 5 folds)')
print(f'Scoring         : F1-macro (adapte au desequilibre)')
print(f'Meilleur modele : Random Forest')
print(f'Accuracy        : {acc_opt:.4f}')
print(f'F1-macro        : {f1_opt:.4f}')
print(f'Precision-macro : {prec_opt:.4f}')
print(f'Recall-macro    : {rec_opt:.4f}')
print(f'ROC-AUC         : {auc_opt:.4f}')
print('=' * 62)
print('Modele sauvegarde => src/best_model.pkl')